# 05 — PCA on Parcel Trajectories

For each subject × task × contrast, build a **(5 sessions × 400 parcels)** matrix,
mean-centre each parcel across sessions, and decompose with PCA (4 components).

- **PC scores** (5×4): how each session loads on each principal component
- **PC loadings** (400×4): which parcels drive each component

PC1 sign is flipped if necessary so that a positive slope across sessions is positive.

Then compute cross-subject RSMs: for each PC, how similar are subjects' parcel loading patterns?

**Outputs (all in `processed_data/`):**
- `surface_pca_results.pkl`

**Figures (in `figures/pca/`):**
- PC loading brain maps per subject × task × contrast
- RSM heatmaps

In [ ]:
import sys
import pickle
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tempfile, os
from pathlib import Path
from collections import defaultdict
from sklearn.decomposition import PCA
from nilearn import plotting, datasets

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import (
    TASKS, CONTRASTS, SUBJECTS, RT_CONTRAST,
    N_ENCOUNTERS, N_VERTICES_PER_HEMI, DATA_DIR, FIGURES_DIR,
    get_schaefer_surface_labels,
)

with open(DATA_DIR / 'surface_parcel_indiv.pkl', 'rb') as f:
    parcel_dict = pickle.load(f)
with open(DATA_DIR / 'surface_indiv_slopes.pkl', 'rb') as f:
    indiv_slopes = pickle.load(f)

labels, parcel_names = get_schaefer_surface_labels()
fsavg6 = datasets.fetch_surf_fsaverage('fsaverage6')

FIG_DIR = FIGURES_DIR / 'pca'
FIG_DIR.mkdir(parents=True, exist_ok=True)

N_COMPONENTS = 4
print('Loaded.')

In [ ]:
def construct_pca_matrix(parcel_dict, subject, task, contrast, n_sessions=N_ENCOUNTERS):
    """
    Build (n_sessions × n_parcels) matrix from parcel activations.
    Missing sessions are imputed with the parcel's mean across available sessions.
    Returns Y (n_sessions, n_parcels) and missing_mask (bool, True = imputed).
    """
    enc_data = parcel_dict.get(subject, {}).get(task, {}).get(contrast, {})
    if not enc_data:
        return None, None

    n_parcels = len(parcel_names)
    Y             = np.full((n_sessions, n_parcels), np.nan)
    missing_mask  = np.zeros((n_sessions, n_parcels), dtype=bool)

    for ek, df in enc_data.items():
        sess_idx = int(ek) - 1   # 0-indexed
        if sess_idx < n_sessions:
            Y[sess_idx, :] = df['activation'].values

    # Impute missing rows with parcel means
    for p_idx in range(n_parcels):
        col = Y[:, p_idx]
        missing = np.isnan(col)
        if missing.any() and (~missing).any():
            parcel_mean = np.nanmean(col)
            Y[missing, p_idx] = parcel_mean
            missing_mask[missing, p_idx] = True

    pct_missing = missing_mask.mean() * 100
    return Y, missing_mask, pct_missing


def run_session_pca(Y, n_components=N_COMPONENTS):
    """
    Mean-centre Y column-wise, then PCA.
    Returns dict with pc_scores (5, n_comp), pc_loadings (400, n_comp),
    explained_variance_ratio, Y_centered, parcel_means.
    """
    parcel_means = Y.mean(axis=0)         # (400,)
    Y_centered   = Y - parcel_means       # (5, 400)

    pca       = PCA(n_components=n_components)
    pc_scores = pca.fit_transform(Y_centered)   # (5, n_comp)

    return {
        'pc_scores':              pc_scores,
        'pc_loadings':            pca.components_.T,   # (400, n_comp)
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'Y_centered':             Y_centered,
        'parcel_means':           parcel_means,
        'pca_object':             pca,
    }

In [ ]:
def slopes_to_vertices(slope_array, labels, n_vertices=81924):
    vertex_data = np.zeros(n_vertices)
    for i, val in enumerate(slope_array):
        vertex_data[labels == (i + 1)] = val
    return vertex_data


def plot_pc_loading_brain(loading_array, pc_idx, task, contrast, subject, save_path):
    """Render 4 surface views of a PC loading map and save."""
    vertex_data = slopes_to_vertices(loading_array, labels)
    lh = vertex_data[:N_VERTICES_PER_HEMI]
    rh = vertex_data[N_VERTICES_PER_HEMI:]
    vmax = np.percentile(np.abs(vertex_data[vertex_data != 0]), 98) if (vertex_data != 0).any() else 1.0

    views = [
        (fsavg6.pial_left,  lh, fsavg6.sulc_left,  'lateral', 'LH lateral'),
        (fsavg6.pial_left,  lh, fsavg6.sulc_left,  'medial',  'LH medial'),
        (fsavg6.pial_right, rh, fsavg6.sulc_right, 'lateral', 'RH lateral'),
        (fsavg6.pial_right, rh, fsavg6.sulc_right, 'medial',  'RH medial'),
    ]

    tmpfiles = []
    for mesh, data, bg, view, label in views:
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False); tmp.close()
        plotting.plot_surf_stat_map(
            mesh, data, bg_map=bg, view=view,
            cmap='RdBu_r', vmax=vmax, colorbar=True,
            title=label, output_file=tmp.name,
        )
        plt.close('all')
        tmpfiles.append(tmp.name)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, tmpf in zip(axes.flat, tmpfiles):
        ax.imshow(mpimg.imread(tmpf)); ax.axis('off')
    title = f'PC{pc_idx+1} loadings | {subject} | {task} | {contrast}'
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()
    for f in tmpfiles:
        os.unlink(f)

In [ ]:
# Main PCA computation loop
pca_results = defaultdict(lambda: defaultdict(dict))
sessions    = np.arange(1, N_ENCOUNTERS + 1)

loadings_dir = FIG_DIR / 'pc_loadings'
loadings_dir.mkdir(parents=True, exist_ok=True)

for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for subject in SUBJECTS:
            result = construct_pca_matrix(parcel_dict, subject, task, contrast)
            if result[0] is None:
                continue
            Y, missing_mask, pct_missing = result

            res = run_session_pca(Y)

            # Flip PC1 sign so that Spearman r with session order is positive
            pc1 = res['pc_scores'][:, 0]
            r_val, _ = stats.spearmanr(sessions, pc1)
            if r_val < 0:
                res['pc_scores'][:, 0]   *= -1
                res['pc_loadings'][:, 0] *= -1

            res['missing_mask']     = missing_mask
            res['percent_missing']  = pct_missing
            res['Y_raw']            = Y
            pca_results[task][contrast][subject] = res

            # Brain map for PC1 and PC2
            for pc_idx in range(min(2, N_COMPONENTS)):
                out = loadings_dir / f'{subject}_{task}_{contrast}_PC{pc_idx+1}.png'
                plot_pc_loading_brain(
                    res['pc_loadings'][:, pc_idx], pc_idx, task, contrast, subject, out
                )
        print(f'  Done: {task} | {contrast}')

print('PCA complete.')

In [ ]:
# Save PCA results (without pca_object — not picklable in all environments)
pca_save = {}
for task in pca_results:
    pca_save[task] = {}
    for contrast in pca_results[task]:
        pca_save[task][contrast] = {}
        for subj, res in pca_results[task][contrast].items():
            pca_save[task][contrast][subj] = {k: v for k, v in res.items() if k != 'pca_object'}

with open(DATA_DIR / 'surface_pca_results.pkl', 'wb') as f:
    pickle.dump(pca_save, f)
print(f'Saved to {DATA_DIR / "surface_pca_results.pkl"}')

In [ ]:
# Cross-subject RSM — for each task/contrast/PC, correlate parcel loading vectors across subjects
rsm_dir = FIG_DIR / 'rsm'
rsm_dir.mkdir(parents=True, exist_ok=True)

rsm_results = defaultdict(lambda: defaultdict(dict))

for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        subj_with_data = [
            s for s in SUBJECTS
            if pca_results.get(task, {}).get(contrast, {}).get(s) is not None
        ]
        if len(subj_with_data) < 2:
            continue

        n_subj = len(subj_with_data)
        for pc_idx in range(N_COMPONENTS):
            rsm = np.zeros((n_subj, n_subj))
            for i, s1 in enumerate(subj_with_data):
                for j, s2 in enumerate(subj_with_data):
                    l1 = pca_results[task][contrast][s1]['pc_loadings'][:, pc_idx]
                    l2 = pca_results[task][contrast][s2]['pc_loadings'][:, pc_idx]
                    rsm[i, j] = np.corrcoef(l1, l2)[0, 1]
            rsm_results[task][contrast][f'PC{pc_idx+1}'] = rsm

        # Plot RSM heatmap for PC1
        fig, axes = plt.subplots(1, N_COMPONENTS, figsize=(4 * N_COMPONENTS, 4))
        for pc_idx, ax in enumerate(axes):
            rsm = rsm_results[task][contrast][f'PC{pc_idx+1}']
            sns.heatmap(rsm, ax=ax, vmin=-1, vmax=1, cmap='RdBu_r', square=True,
                        xticklabels=subj_with_data, yticklabels=subj_with_data,
                        annot=True, fmt='.2f', annot_kws={'size': 7})
            ax.set_title(f'PC{pc_idx+1}  mean r={np.triu(rsm,1).mean():.2f}')
        fig.suptitle(f'Loading RSM | {task} | {contrast}', fontsize=10)
        plt.tight_layout()
        plt.savefig(rsm_dir / f'{task}_{contrast}_rsm.png', dpi=100, bbox_inches='tight')
        plt.close()

print(f'RSM figures saved to {rsm_dir}')

In [ ]:
# Summary: explained variance and PC1 correlation with slopes across subjects
from scipy.stats import pearsonr

print('Explained variance ratio (mean across subjects)\n')
for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        subj_data = [
            pca_results[task][contrast][s]
            for s in SUBJECTS
            if pca_results.get(task, {}).get(contrast, {}).get(s) is not None
        ]
        if not subj_data:
            continue
        mean_ev = np.mean([r['explained_variance_ratio'] for r in subj_data], axis=0)
        ev_str  = '  '.join(f'PC{i+1}={v:.2%}' for i, v in enumerate(mean_ev))

        # PC1 loading vs slope correlation (averaged across subjects)
        pc1_slope_corrs = []
        for s in SUBJECTS:
            res  = pca_results.get(task, {}).get(contrast, {}).get(s)
            if res is None:
                continue
            indiv = indiv_slopes.get(s, {}).get(task, {}).get(contrast, {})
            slopes_arr = np.array([indiv.get(p, {}).get('beta_slope', np.nan) for p in parcel_names])
            valid = ~np.isnan(slopes_arr)
            if valid.sum() >= 10:
                r, _ = pearsonr(res['pc_loadings'][:, 0][valid], slopes_arr[valid])
                pc1_slope_corrs.append(r)
        r_str = f'  PC1~slope r={np.mean(pc1_slope_corrs):.3f}' if pc1_slope_corrs else ''

        print(f'  {task} | {contrast}: {ev_str}{r_str}')